# 08 — Drought Indices: SPI and SPEI from CHIRPS Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ByMaxAnjos/LCZ4py/blob/main/notebooks/local/08_drought_indices_spi_spei.en.ipynb)

**Learning objective**: compute the Standardized Precipitation Index (SPI) and the Standardized Precipitation-Evapotranspiration Index (SPEI) at multiple timescales (1, 3, 6, and 12 months) from a monthly precipitation time series. By the end of this notebook you will know how to assemble the input table that `lcz_climate_compute_spi` and `lcz_climate_compute_spei` expect, how to interpret the resulting drought classifications, and why SPEI — by folding in evapotranspiration — is more sensitive to warming-amplified drought than SPI alone.

## Why standardized drought indices, and why multiple timescales

SPI (McKee et al. 1993) is the most widely used meteorological drought index in the world: it turns accumulated precipitation into a dimensionless number, comparable across climates and regions, by fitting a gamma distribution to the local historical record and then standardizing the result to a standard normal distribution (mean 0, standard deviation 1). An SPI of -1.5 means the same thing in Fortaleza, Berlin, or Manaus: a precipitation anomaly that historically occurs with the same relative rarity, even though the absolute rainfall regimes are completely different.

The timescale (the number of accumulated months before standardizing) matters as much as the index itself. **Short scales (1-3 months)** respond quickly — they reflect soil-moisture content and are most relevant for agricultural drought: a crop feels a rainfall shortfall within weeks, not years. **Long scales (6-12 months or more)** smooth out short-term variability and capture the accumulated water balance that fills (or drains) reservoirs, aquifers, and rivers — so-called hydrological drought. That is why `lcz_climate_compute_spi` and `lcz_climate_compute_spei` compute four scales simultaneously by default (1, 3, 6, 12 months): no single scale tells the whole story of a drought.

SPEI (Vicente-Serrano et al. 2010) addresses a well-known limitation of SPI: it completely ignores the atmosphere's evaporative demand. In a warming world, a city can receive the same rainfall it always has and still slide into severe water deficit because potential evapotranspiration (PET) rose along with temperature. SPEI replaces raw precipitation with the water balance D = precipitation − PET before standardizing, making it sensitive to *temperature-driven* drought that SPI simply cannot see. This is particularly relevant for LCZ4py's urban-climate focus: urban heat (the heat island) raises local evaporative demand beyond what the surrounding rural landscape experiences, so a city can show a more severe SPEI than SPI even under normal rainfall — a direct signal that the urban climate is amplifying local water stress rather than merely reflecting a regional rainfall shortfall.

Running these indices on a CHIRPS precipitation series **cropped to the city's LCZ map footprint** (rather than a single station or a generic regional average) ties this notebook back into the rest of the LCZ4py workflow: the same spatial extent used to download the LCZ map (`lcz_get_map`), extract morphological parameters, and compute spectral indices is now the area over which we quantify drought risk — every product in the package consistently describes the same "city".

Both functions follow LCZ4py's shared climate-function signature: input and output `df` are `pandas.DataFrame` (not raster, not polars) — the one "purely tabular" exception in this tutorial series, because SPI/SPEI operate on an aggregated time series, not pixel by pixel.

## Install LCZ4py

In [1]:
!pip -q install "LCZ4py[all] @ git+https://github.com/ByMaxAnjos/LCZ4py.git" 


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: /Applications/QGIS.app/Contents/MacOS/bin/python3 -m pip install --upgrade pip
ERROR: Package 'lcz4py' requires a different Python: 3.9.5 not in '>=3.10'


## Step 1 — Assemble the input time series from CHIRPS

No dataset already bundled in the repo (not even the Berlin station CSV, which only has air temperature) comes with ready-made monthly precipitation — this is the least "plug-and-play" step of the whole tutorial series, so let's build it carefully, explaining each transformation.

1. Download a small city's LCZ map (`lcz_get_map`, notebook `01`) to define the spatial extent.
2. Use `lcz_grid_chirps(x, resolution="monthly", years=[...])` (notebook `general/07`) to download monthly CHIRPS precipitation cropped to that extent. It returns an `LCZGridResult` whose `.array` has shape `(n_months, H, W)` (mm/month, NaN outside the valid area) and whose `.bands[i]` is the ISO date of each month.
3. Since SPI/SPEI operate on **one time series per unit** (the signature expects a `code_muni` column — inherited from the sibling `climasus4r` package's origin, which processes Brazilian municipalities — but here a constant identifier for "this city" is enough), we collapse each monthly band down to a single number: the spatial mean precipitation over every valid pixel in the city. That yields exactly one monthly time series, which is the expected input shape.
4. Assemble the `pandas.DataFrame` with the required columns: `code_muni` (constant), `date` (monthly, datetime dtype), and the precipitation column, whose name must match the functions' `var`/`rain_var` parameter — we'll use the default name `rainfall_chirps_mm` so we don't need to override that argument.
5. SPEI, when `pet_method="thornthwaite"`, also needs a monthly mean temperature column (`temp_var`, default `tair_dry_bulb_c`) to derive potential evapotranspiration internally (see `_thornthwaite_pet` in the source). Since we don't have ERA5 or a long-running monthly-temperature station ready for this city in this notebook, we **synthesize** a plausible monthly temperature series (a sinusoidal seasonal cycle plus noise) purely for didactic purposes. In a real study this column would come from observed data — e.g. `lcz_grid_era5`/`lcz_grid_era5_global` (variable `t2m`, already covered in notebook `general/07`) aggregated the same way we aggregate CHIRPS below, or a local weather station.

In [2]:
from LCZ4py import lcz_get_map, lcz_grid_chirps

map_path = lcz_get_map(city="Vaduz", isave_map=True)
print(map_path)

06:42:44 - LCZ4py._internal._lcz_map_engine - INFO - Loading clipped map from local cache.


06:42:44 - rasterio._env - WARNING - CPLE_IllegalArg in tmpljt8rhb2.tif: BLOCKXSIZE can only be used with TILED=YES


06:42:44 - LCZ4py._internal._lcz_map_engine - INFO - Saved map to LCZ4r_output/lcz_map.tif


/Users/co2map/.lcz4r_cache/clipped_a8db9faa23e32c3b.tif


The `lcz_get_map` call above downloads (or reuses from cache) Vaduz's LCZ map — chosen for its small footprint, which keeps CHIRPS downloads fast for this worked example. In a real study, just swap `city=` for the city you care about.

In [3]:
import numpy as np
import pandas as pd

years = [2022, 2023]  # 2 years ~ 24 monthly data points (keeps the download fast for this example)
chirps = lcz_grid_chirps(map_path, resolution="monthly", years=years, isave=False)

print(chirps.array.shape, len(chirps.bands))
print(chirps.bands[:3], "...", chirps.bands[-3:])

Using cached CHIRPS stack.
Complete: CHIRPS stack with 24 band(s), cropped to the LCZ map.
(24, 120, 131) 24
['2022-01-01', '2022-02-01', '2022-03-01'] ... ['2023-10-01', '2023-11-01', '2023-12-01']


`chirps.array` has shape `(n_months, H, W)`; each `chirps.bands[i]` is the ISO date (first day of month) of that band. Now we collapse each band to its spatial mean (ignoring NaN) to get a single monthly time series representing the whole city.

In [4]:
monthly_rain = np.nanmean(chirps.array.reshape(chirps.array.shape[0], -1), axis=1)
dates = pd.to_datetime(chirps.bands)

# Synthetic temperature series (seasonal cycle + noise) — REPLACE with real
# observed data (e.g. ERA5 t2m via lcz_grid_era5_global) in a real study.
rng = np.random.default_rng(42)
month_num = dates.month.to_numpy()
synthetic_temp = 10 + 8 * np.sin(2 * np.pi * (month_num - 4) / 12) + rng.normal(0, 0.8, len(dates))

df = pd.DataFrame({
    "code_muni": "vaduz",       # constant identifier: a single "unit" (the city)
    "date": dates,
    "rainfall_chirps_mm": monthly_rain,
    "tair_dry_bulb_c": synthetic_temp,
})
df = df.sort_values("date").reset_index(drop=True)
df.head()

,code_muni,date,rainfall_chirps_mm,tair_dry_bulb_c
0,vaduz,2022-01-01,37.658882,2.243774
1,vaduz,2022-02-01,111.390411,2.239809
2,vaduz,2022-03-01,34.109623,6.600361
3,vaduz,2022-04-01,94.559219,10.752452
4,vaduz,2022-05-01,111.366608,12.439172


This `df` — with `code_muni`, a monthly `date`, the precipitation column, and the temperature column — is exactly the tabular shape `lcz_climate_compute_spi` and `lcz_climate_compute_spei` expect. Note that with only 2 years (~24 months) of data — deliberately small to keep CHIRPS downloads fast for this demo notebook — this example is purely didactic: the WMO recommends at least 30 years of calibration for a robust SPI gamma fit; we use a lower `min_n` below just so the demonstration runs (with 24 months and a 12-month scale, only 13 rolling-sum values are even possible).

## Step 2 — `lcz_climate_compute_spi`: SPI at multiple timescales

Computes SPI for every scale in `scales` (default `(1, 3, 6, 12)` months). For each scale *s* and each unit (`code_muni`): an *s*-month rolling sum of precipitation, a zero-inflated gamma fit to the calibration period (`ref_start`/`ref_end`, or the whole period if `None`) via method of moments, then standardization to a standard normal. `min_n` (default 24) is the minimum number of non-NA calibration values — below that the function leaves the index as NaN for that unit/scale. The return value is the input `df` with added `spi_{s}mo` columns (one per scale).

Default SPI classification: ≥2.0 extremely wet, 1.5–1.99 very wet, 1.0–1.49 moderately wet, -0.99..0.99 near normal, -1..-1.49 moderately dry (D1), -1.5..-1.99 severely dry (D2), ≤-2.0 extremely dry (D3-D4).

In [5]:
from LCZ4py import lcz_climate_compute_spi

spi_df = lcz_climate_compute_spi(
    df,
    var="rainfall_chirps_mm",
    scales=(1, 3, 6, 12),
    min_n=12,  # lowered only because this example has few years; use >=24 (ideally 30 years) in production
)
spi_df[["date", "rainfall_chirps_mm", "spi_1mo", "spi_3mo", "spi_6mo", "spi_12mo"]].tail(12)

Computing SPI (Standardized Precipitation Index)
Computing 1-month scale -> spi_1mo...
Computing 3-month scale -> spi_3mo...
Computing 6-month scale -> spi_6mo...
Computing 12-month scale -> spi_12mo...
Done: 24 rows, 0 NA(s) in 'spi_1mo'.


,date,rainfall_chirps_mm,spi_1mo,spi_3mo,spi_6mo,spi_12mo
12,2023-01-01,61.398342,-1.160628,-1.001060,-0.385862,-0.980131
13,2023-02-01,54.622860,-1.328720,-1.322701,-1.110639,-1.361164
14,2023-03-01,140.415771,0.212960,-1.158153,-1.224711,-0.655919
15,2023-04-01,160.115967,0.466478,-0.294653,-1.000358,-0.237968
16,2023-05-01,147.522842,0.306988,0.379470,-0.655228,-0.012553
17,2023-06-01,85.517754,-0.652131,-0.006111,-0.873651,-0.563166
18,2023-07-01,226.610779,1.195349,0.456852,0.009596,-0.114808
19,2023-08-01,350.782135,2.252654,1.644239,1.331577,0.895852
20,2023-09-01,83.560089,-0.689328,1.634020,1.097577,0.449725
21,2023-10-01,139.885239,0.205815,1.160299,1.012329,0.550707


Each `spi_{s}mo` column is independent of the others: a given month can be in severe drought on the 1-month scale (that specific month's rainfall was near zero) yet close to normal on the 12-month scale (if earlier months compensated). Reading all four scales side by side is what lets you tell a passing "dry month" apart from a structural hydrological drought building up over the year. `NaN` values at the start of the series are expected — the *s*-month rolling sum only exists from the *s*-th month onward, and the gamma calibration requires `min_n` valid observations.

## Step 3 — `lcz_climate_compute_spei`: SPEI folding in evapotranspiration

Same multi-timescale mechanics as SPI, but instead of standardizing raw precipitation it standardizes the water balance D = precipitation − PET (an *s*-month rolling sum), using the empirical Hazen plotting position (`p = (rank-0.5)/n`) rather than a gamma fit — valid for any shape of D, including negative values (the water balance can go negative, unlike precipitation).

PET (potential evapotranspiration) can come from two sources, controlled by `pet_method`:
- `pet_method="column"` (default): reads an already-computed PET column, `pet_var` (default `"pet_mm"`) — useful if you already have PET from a product like ERA5-Land or an external Penman-Monteith calculation.
- `pet_method="thornthwaite"`: derives PET internally from `temp_var` (monthly mean temperature, default `"tair_dry_bulb_c"`) using the Thornthwaite (1948) method — a simple temperature-only method, with no day-length/latitude correction (reasonable for tropical regions where day length varies little; less accurate at high latitudes with very long/short summer/winter days). That's the path we'll use here, since we only have the synthetic temperature column.

In [6]:
from LCZ4py import lcz_climate_compute_spei

spei_df = lcz_climate_compute_spei(
    spi_df,  # reuse the df that already carries the spi_* columns
    rain_var="rainfall_chirps_mm",
    pet_method="thornthwaite",
    temp_var="tair_dry_bulb_c",
    scales=(1, 3, 6, 12),
    min_n=12,
)
spei_df[["date", "rainfall_chirps_mm", "tair_dry_bulb_c", "spi_12mo", "spei_12mo"]].tail(12)

Computing SPEI (Standardized Precipitation-Evapotranspiration Index)
Computing 1-month scale -> spei_1mo...
Computing 3-month scale -> spei_3mo...
Computing 6-month scale -> spei_6mo...
Computing 12-month scale -> spei_12mo...
Done: 24 rows, 0 NA(s) in 'spei_1mo'.


,date,rainfall_chirps_mm,tair_dry_bulb_c,spi_12mo,spei_12mo
12,2023-01-01,61.398342,2.052825,-0.980131,-0.869424
13,2023-02-01,54.622860,3.973590,-1.361164,-1.768825
14,2023-03-01,140.415771,6.374007,-0.655919,-0.615141
15,2023-04-01,160.115967,9.312566,-0.237968,-0.194028
16,2023-05-01,147.522842,14.295001,-0.012553,0.194028
17,2023-06-01,85.517754,16.161097,-0.563166,-0.395725
18,2023-07-01,226.610779,18.702760,-0.114808,0.000000
19,2023-08-01,350.782135,16.888263,0.895852,0.869424
20,2023-09-01,83.560089,13.852110,0.449725,0.395725
21,2023-10-01,139.885239,9.455256,0.550707,0.615141


Compare `spi_12mo` with `spei_12mo` for the same month: if `spei_12mo` is consistently more negative than `spi_12mo`, that's the signature of a *temperature-amplified* drought — rainfall alone would not be classified as this severe, but elevated evaporative demand (higher temperatures pulling more water out of soil and vegetation) pushes the water balance into a deeper deficit. In hot cities — especially under the urban heat island effect that LCZ4py characterizes via `lcz_uhi_intensity`/`lcz_get_ucp` — this SPI-vs-SPEI gap tends to be more pronounced than in the surrounding countryside, because local excess heat raises PET locally with no change in regional rainfall at all. That is exactly the kind of evidence — drought "invisible" to SPI but captured by SPEI — that motivates running both indices, and running them over the specific spatial extent derived from the city's LCZ map rather than a generic regional average that dilutes the urban signal.

## Conclusion, and the end of the tutorial series

This notebook closes out the 8-notebook `local/` series arc: we started with LCZ-stratified time series (`01`), moved through temperature anomalies (`02`), urban heat island intensity (`03`), geostatistical interpolation via kriging (`04`) and machine-learning interpolation (`05`), temporal climate metrics like diurnal temperature range and degree-days (`06`), thermal comfort and anthropogenic heat (`07`), and finished here with standardized multi-timescale drought indices (`08`).

Combined with the `general/` series — LCZ map acquisition, visualization and area statistics, morphological parameters, remote sensing (LST via Planetary Computer), spectral indices, urban canopy parameters, and gridded climate/environmental data — you now have all 15 pieces of a complete LCZ-based urban climate workflow: from downloading your city's classification map all the way to quantifying drought, heat, thermal comfort, and spatial temperature variability over that same consistent geographic extent. The natural next step is applying this workflow to your own city — swapping `city="Vaduz"` (or whichever example city was used throughout the series) for your real study area, and the example stations/grids for your own observational data.

**Previous**: `07_thermal_comfort_anthropogenic_heat` (thermal comfort and anthropogenic heat).
**Next**: none — this is the final notebook in the `local/` series, and in the complete 15-notebook `general/` + `local/` LCZ4py tutorial arc.